# [센서 실습 12] 이산화탄소 센서 – MH-Z19B (NDIR)

적외선(NDIR) 방식 CO₂ 센서 **MH-Z19B**에 UART 명령을 보내고 9바이트 응답을 해석하여, 교실 수업 중 CO₂ 농도 변화와 환기 시점을 판단합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
CO₂ 분자는 **4.26µm 파장의 적외선**을 흡수합니다. NDIR(비분산 적외선) 센서는 이 성질을 이용합니다.
- 관 한쪽에서 적외선을 쏘고 반대쪽에서 세기를 측정 → CO₂가 많을수록 **도착하는 빛이 약해짐**
- 화학 반응식 센서보다 정확하고 수명이 김
- 통신 방식: 라즈베리파이가 **읽기 명령 9바이트**를 보내면, 센서가 **응답 9바이트**를 돌려줌

| 구분 | 바이트 |
|---|---|
| 읽기 명령 | `FF 01 86 00 00 00 00 00 79` |
| 응답 | `FF 86 [상위] [하위] … [체크섬]` → 농도 = 상위 × 256 + 하위 |

체크섬 = `0xFF − (1~7번 바이트 합의 하위 8비트) + 1`

### 부품 사양
| 항목 | 내용 |
|---|---|
| 측정 범위 | 0 ~ 5,000 ppm |
| 정확도 | ±(50 ppm + 측정값의 5%) |
| 예열 | 3분 |
| 응답 시간 | 120초 이내 (90% 반응) |
| 전원 | 4.5 ~ 5.5V |
| 통신 | UART 9600bps (3.3V 레벨) |

## 2. 라즈베리파이 연결 방법
**준비물**: MH-Z19B, 점퍼선 4개

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| MH-Z19B VCC | 5V (2번 핀) |  |
| MH-Z19B GND | GND (6번 핀) |  |
| MH-Z19B TX | GPIO15 / RXD (10번 핀) | 센서 송신 → Pi 수신 |
| MH-Z19B RX | GPIO14 / TXD (8번 핀) | Pi 송신 → 센서 수신 |



**인터페이스 활성화**
- **UART**: `sudo raspi-config` → Interface Options → **Serial Port** → 로그인 셸: 아니요, 시리얼 하드웨어: 예

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install pyserial
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
명령·응답의 체크섬 계산과 응답 해석 함수입니다.

In [ ]:
READ_CMD = bytes([0xFF, 0x01, 0x86, 0x00, 0x00, 0x00, 0x00, 0x00, 0x79])

def mhz19_checksum(packet):
    return (0xFF - (sum(packet[1:8]) & 0xFF) + 1) & 0xFF

def parse_mhz19(resp):
    """9바이트 응답 → CO2 ppm (오류면 None)"""
    if resp is None or len(resp) != 9 or resp[0] != 0xFF or resp[1] != 0x86:
        return None
    if mhz19_checksum(resp) != resp[8]:
        return None
    return resp[2] * 256 + resp[3]

print("읽기 명령 체크섬:", hex(mhz19_checksum(READ_CMD)))   # 0x79

### 4-2. 실제 센서 클래스
명령을 보내고 9바이트를 받아 해석합니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealMHZ19:
    def __init__(self, port="/dev/serial0"):
        import serial
        self.ser = serial.Serial(port, baudrate=9600, timeout=1)

    def read(self):
        self.ser.reset_input_buffer()
        self.ser.write(READ_CMD)
        resp = self.ser.read(9)
        return {"co2_ppm": parse_mhz19(resp)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
가상 센서는 교실 공기를 **상자 모형(box model)** 으로 계산합니다.

$$\frac{dC}{dt} = \frac{n \cdot G}{V} - \lambda\,(C - C_{out})$$

| 기호 | 가정한 값 | 근거 |
|---|---|---|
| 교실 부피 $V$ | 180 ㎥ (60㎡ × 높이 3m) | 일반 교실 |
| 학생 수 $n$ | 수업 중 25명, 쉬는 시간 5명 | 학급 인원 |
| 1인 배출량 $G$ | 0.018 ㎥/h (앉아서 공부) | 성인 안정 시 호흡 |
| 환기율 $\lambda$ | 창문 닫음 0.5회/h, 쉬는 시간 창문 열면 6회/h | 실내 환기 연구의 일반 값 |
| 바깥 농도 $C_{out}$ | 420 ppm | 현재 대기 CO₂ |
| 시간표 | 9:00~9:50 수업 / 9:50~10:00 휴식 / 10:00~10:50 수업 / 10:50~11:00 휴식 / 11:00~12:00 수업(환기 없음) | |

계산한 농도는 **실제 응답과 같은 9바이트(체크섬 포함)** 로 바꿔 해석기에 넘깁니다.

In [ ]:
class VirtualMHZ19:
    V, G, C_OUT = 180.0, 0.018, 420.0

    def __init__(self, dt=60, seed=12):
        self.dt, self.t, self.c = dt, 0.0, 450.0
        self.rng = np.random.default_rng(seed)

    def _schedule(self, minute):
        if minute < 50 or 60 <= minute < 110 or minute >= 120:
            return 25, 0.5                    # 수업 중 (창문 닫음)
        return 5, 6.0                         # 쉬는 시간 (창문 열고 환기)

    def read(self):
        n, lam = self._schedule(self.t / 60)
        dt_h = self.dt / 3600
        self.c += (n * self.G / self.V * 1e6 - lam * (self.c - self.C_OUT)) * dt_h
        ppm = int(np.clip(self.c + self.rng.normal(0, 20), 0, 5000))
        resp = [0xFF, 0x86, ppm >> 8, ppm & 0xFF, 0x47, 0x00, 0x00, 0x00]
        resp.append(mhz19_checksum(resp + [0]))
        self.t += self.dt
        return {"co2_ppm": parse_mhz19(bytes(resp))}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualMHZ19(dt=60)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
1분 간격으로 9시부터 3시간(180회) 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 180               # 측정 횟수
INTERVAL = 60               # 측정 간격(초)

sensor = RealMHZ19() if USE_REAL_SENSOR else VirtualMHZ19(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
실내공기질 관리 기준(다중이용시설)에서 **CO₂ 유지기준은 1,000 ppm**입니다.

| CO₂ | 상태 |
|---|---|
| 1,000 ppm 미만 | 양호 |
| 1,000 ~ 2,000 ppm | 환기 필요 (졸음·집중력 저하) |
| 2,000 ppm 이상 | 즉시 환기 (두통 가능) |

In [ ]:
df["clock"] = [f"{9 + int(s // 3600)}:{int(s % 3600 // 60):02d}" for s in df["time_s"]]
df["state"] = pd.cut(df["co2_ppm"], [0, 1000, 2000, 99999], labels=["양호", "환기 필요", "즉시 환기"], right=False)
over = df[df["co2_ppm"] >= 1000]
print(f"처음 1000ppm 초과: {over['clock'].iloc[0]}  (수업 시작 {over['time_s'].iloc[0] / 60:.0f}분 후)")
print(f"최고 농도: {df['co2_ppm'].max()} ppm ({df.loc[df['co2_ppm'].idxmax(), 'clock']})")
print("상태별 시간(분):", (df["state"].value_counts(sort=False) * INTERVAL / 60).to_dict())
print("\n→ 50분 수업 동안 창문을 닫아 두면 기준을 넘습니다. 수업 중간에 짧게 환기하는 것이 좋습니다.")

## 8. 시각화

In [ ]:
plt.figure(figsize=(10, 4.5))
m = df["time_s"] / 60
for a, b in [(50, 60), (110, 120)]:
    plt.axvspan(a, b, color="lightgray"); plt.text(a + 1, 300, "휴식\n환기", fontsize=8)
plt.plot(m, df["co2_ppm"], "k-", lw=1.5, label="CO2")
plt.axhline(1000, color="gray", ls="--", label="유지기준 1000 ppm")
plt.axhline(2000, color="gray", ls=":", label="즉시 환기 2000 ppm")
plt.xticks(range(0, 181, 30), ["9:00", "9:30", "10:00", "10:30", "11:00", "11:30", "12:00"])
plt.title("MH-Z19B 교실 CO2 농도"); plt.ylabel("ppm"); plt.grid(alpha=0.3); plt.legend(loc="upper left")
plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "CO2센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "MH-Z19B (NDIR)",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/co2", payload)
# client.disconnect()

## 직접 해 보기
- 학생 수를 15명으로 줄이면 기준을 넘는 시간이 어떻게 달라지나요?
- 수업 중 환기율을 2회/h(창문 조금 열기)로 바꾸면 어떻게 되나요?
- 1000 ppm을 넘으면 '창문을 열어 주세요' 메시지를 MQTT로 보내는 JSON을 만들어 보세요.